# 09b — XGBoost, LightGBM y CatBoost
**Referencias:** Chen & Guestrin (2016) — XGBoost paper · Ke et al. (2017) — LightGBM paper · Prokhorenkova et al. (2018) — CatBoost paper

Los tres son implementaciones de **Gradient Boosting** sobre árboles, pero con distintas estrategias de construcción y regularización.

## Gradient Boosting — idea central (ESL 10.3)
Se construye un ensemble aditivo de árboles $F_M(x) = \sum_{m=1}^M \eta \cdot h_m(x)$ donde cada árbol $h_m$ se ajusta sobre los **residuales negativos del gradiente** de la pérdida anterior:
$$h_m = \underset{h}{\arg\min} \sum_{i=1}^N \left[ -g_i \cdot h(x_i) \right], \quad g_i = \frac{\partial L(y_i, F_{m-1}(x_i))}{\partial F_{m-1}(x_i)}$$

## Diferencias clave

| | **XGBoost** | **LightGBM** | **CatBoost** |
|---|---|---|---|
| Split | Level-wise | Leaf-wise (GOSS) | Symmetric (oblivious) |
| Regularización | L1 + L2 en objetivo | L1 + L2 | L2 implícita |
| Categóricas | Encoding manual | Encoding manual | Nativas (ordered boosting) |
| Velocidad | Rápido | Muy rápido (datasets grandes) | Más lento, menos tuning |
| Overfitting | `eta` + `max_depth` + `subsample` | `num_leaves` + `min_data_in_leaf` | Poco sensible por default |
| API sklearn | `XGBClassifier` | `LGBMClassifier` | `CatBoostClassifier` |

## XGBoost — regularización en el objetivo (paper Chen 2016)
$$\mathcal{L}^{(m)} = \sum_i \left[ g_i f_m(x_i) + \frac{1}{2} h_i f_m^2(x_i) \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2$$
donde $g_i$ y $h_i$ son gradiente y Hessiana de la pérdida, $T$ es el número de hojas y $w_j$ los pesos de cada hoja.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    mean_squared_error, r2_score
)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor, Pool

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# Dataset: comportamiento de usuarios en plataforma SaaS
n = 3000
sessions       = np.random.randint(1, 30, n)
time_on_site   = np.random.uniform(30, 900, n)
pages          = np.random.uniform(1, 12, n)
days_since_reg = np.random.randint(0, 60, n)
features_used  = np.random.randint(0, 15, n)
errors_seen    = np.random.poisson(1.5, n)
channel        = np.random.choice(['organic', 'paid', 'email', 'direct', 'referral'], n)
device         = np.random.choice(['mobile', 'desktop', 'tablet'], n)
plan           = np.random.choice(['free', 'trial', 'pro', 'enterprise'], n, p=[0.55, 0.25, 0.15, 0.05])
country        = np.random.choice(['US', 'UK', 'DE', 'MX', 'BR', 'other'], n)

logit = (
    -4 + sessions * 0.10 + time_on_site * 0.0018 + pages * 0.15
    - days_since_reg * 0.05 + features_used * 0.22 - errors_seen * 0.3
    + np.where(channel == 'email', 1.0, 0)
    + np.where(device == 'desktop', 0.5, 0)
    + np.where(plan == 'trial', 1.3, np.where(plan == 'pro', 2.1, np.where(plan == 'enterprise', 3.0, 0)))
    + np.where(country == 'US', 0.4, np.where(country == 'UK', 0.2, 0))
)
prob = 1 / (1 + np.exp(-logit))
converted = (np.random.uniform(0, 1, n) < prob).astype(int)

revenue = (
    80 + sessions * 12 + time_on_site * 0.2 + pages * 8
    + features_used * 18 - days_since_reg * 2 - errors_seen * 10
    + np.where(plan == 'trial', 90, np.where(plan == 'pro', 350, np.where(plan == 'enterprise', 1200, 0)))
    + np.where(device == 'desktop', 50, 0)
    + np.random.normal(0, 70, n)
).clip(0)

df = pd.DataFrame({
    'sessions': sessions, 'time_on_site': time_on_site.round(0),
    'pages': pages.round(2), 'days_since_reg': days_since_reg,
    'features_used': features_used, 'errors_seen': errors_seen,
    'channel': channel, 'device': device, 'plan': plan, 'country': country,
    'converted': converted, 'revenue': revenue.round(2),
})

num_features = ['sessions', 'time_on_site', 'pages', 'days_since_reg', 'features_used', 'errors_seen']
cat_features = ['channel', 'device', 'plan', 'country']

X = df[num_features + cat_features]
y_clf = df['converted']
y_reg = df['revenue']

X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_clf, y_reg, test_size=0.2, random_state=42, stratify=y_clf
)
print(f'Dataset: {n} filas | Tasa conversión: {y_clf.mean():.2%} | Revenue media: ${y_reg.mean():.0f}')

## 1 — Preprocesamiento: estrategias por framework
- **XGBoost / LightGBM**: no soportan strings — requieren encoding manual (ordinal o target encoding)
- **CatBoost**: acepta categóricas como strings directamente — les aplica *ordered target encoding* internamente

In [ ]:
# Preprocessor para XGBoost y LightGBM (OrdinalEncoder — suficiente para GBTs)
preprocessor_ord = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), cat_features),
])

X_train_enc = preprocessor_ord.fit_transform(X_train)
X_test_enc  = preprocessor_ord.transform(X_test)
feat_names  = num_features + cat_features

print('Encoding para XGBoost/LightGBM:')
print(pd.DataFrame(X_train_enc[:3], columns=feat_names).to_string())

# Para CatBoost: pasar los datos sin encoding + índices de columnas categóricas
cat_indices = [X_train.columns.tolist().index(c) for c in cat_features]
print(f'\nCatBoost recibe strings directamente. Índices categóricos: {cat_indices}')

## 2 — XGBoost: hiperparámetros clave

In [ ]:
# Hiperparámetros más importantes de XGBoost:
# n_estimators   : número de árboles (con early stopping, se puede poner alto)
# learning_rate  : shrinkage por árbol (alias: eta) — típico 0.01–0.3
# max_depth      : profundidad máxima — típico 3–8
# subsample      : fracción de filas por árbol — anti-overfitting
# colsample_bytree: fracción de features por árbol — anti-overfitting
# min_child_weight: mínimo de suma de hessiana en hoja — como min_samples_leaf
# gamma (min_split_loss): penalización para hacer un split — mayor = árbol más conservador
# reg_alpha (L1), reg_lambda (L2): regularización explícita sobre pesos de hojas

xgb_configs = [
    ('XGB default',           xgb.XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)),
    ('XGB lr=0.05 deep',      xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                                  subsample=0.8, colsample_bytree=0.8,
                                                  random_state=42, eval_metric='logloss', verbosity=0)),
    ('XGB lr=0.3 shallow',    xgb.XGBClassifier(n_estimators=100, learning_rate=0.3, max_depth=3,
                                                  random_state=42, eval_metric='logloss', verbosity=0)),
    ('XGB regularizado',      xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                                                  subsample=0.7, colsample_bytree=0.7,
                                                  reg_alpha=0.5, reg_lambda=2.0, gamma=0.1,
                                                  random_state=42, eval_metric='logloss', verbosity=0)),
]

print(f'{"Modelo":<25} {"TrainAUC":>10} {"TestAUC":>9} {"F1":>6}')
print('-' * 55)
for name, model in xgb_configs:
    model.fit(X_train_enc, yc_train)
    tr_auc = roc_auc_score(yc_train, model.predict_proba(X_train_enc)[:, 1])
    te_auc = roc_auc_score(yc_test,  model.predict_proba(X_test_enc)[:, 1])
    f1     = f1_score(yc_test, model.predict(X_test_enc))
    print(f'{name:<25} {tr_auc:>10.4f} {te_auc:>9.4f} {f1:>6.4f}')

In [ ]:
# Early Stopping: entrenar con muchos árboles, parar cuando el val_score no mejora
X_tr2, X_val, y_tr2, y_val = train_test_split(X_train_enc, yc_train, test_size=0.15, random_state=0)

xgb_es = xgb.XGBClassifier(
    n_estimators=1000, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    early_stopping_rounds=30,
    eval_metric='auc',
    random_state=42, verbosity=0
)
xgb_es.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)], verbose=False)

print(f'Early stopping en ronda: {xgb_es.best_iteration}')
print(f'Val AUC best:            {xgb_es.best_score:.4f}')
print(f'Test AUC:                {roc_auc_score(yc_test, xgb_es.predict_proba(X_test_enc)[:,1]):.4f}')

# Curva de entrenamiento
evals_result = xgb_es.evals_result()
val_auc = evals_result['validation_0']['auc']

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(val_auc, color='#4361ee', linewidth=1.5, label='Val AUC')
ax.axvline(xgb_es.best_iteration, color='#f72585', linestyle='--', linewidth=2,
           label=f'Early stop: ronda {xgb_es.best_iteration}')
ax.set_xlabel('Número de árboles'); ax.set_ylabel('AUC')
ax.set_title('XGBoost — Early Stopping')
ax.legend(); plt.tight_layout(); plt.show()

## 3 — LightGBM: leaf-wise growth y GOSS
LightGBM crece el árbol **leaf-wise** (expande la hoja con mayor pérdida), no level-wise como XGBoost. Esto lo hace más rápido en datasets grandes pero más propenso a overfitting si `num_leaves` es muy alto.

**GOSS** (Gradient-based One-Side Sampling): descarta instancias con gradiente pequeño (bien aprendidas), conservando las difíciles. Reduce datos sin perder información clave.

Hiperparámetros clave de LightGBM:
- `num_leaves`: controla complejidad (equivalente a `max_depth` pero más granular — regla: `num_leaves < 2^max_depth`)
- `min_data_in_leaf`: mínimo de muestras en hoja — el más importante para anti-overfitting
- `learning_rate` + `n_estimators` + early stopping
- `feature_fraction`, `bagging_fraction`: submuestreo de features y filas

In [ ]:
# LightGBM con early stopping
lgb_model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.05,
    num_leaves=63, min_child_samples=20,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbose=-1
)
lgb_model.fit(
    X_tr2, y_tr2,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
)

print(f'Early stopping en ronda: {lgb_model.best_iteration_}')
print(f'Test AUC: {roc_auc_score(yc_test, lgb_model.predict_proba(X_test_enc)[:,1]):.4f}')
print(f'Test F1:  {f1_score(yc_test, lgb_model.predict(X_test_enc)):.4f}')

# Feature importance
lgb_imp = pd.Series(lgb_model.feature_importances_, index=feat_names).sort_values(ascending=False)
print(f'\nTop 6 features (LightGBM gain):')
print(lgb_imp.head(6).to_string())

## 4 — CatBoost: categóricas nativas y symmetric trees
CatBoost construye árboles **simétricos** (oblivious trees): el mismo split se aplica en todos los nodos del mismo nivel → estructuras más rápidas en inferencia.

Su ventaja principal: no requiere encoding de categóricas — las procesa con *ordered target encoding* que evita data leakage al usar solo el historial previo de cada muestra durante el entrenamiento.

In [ ]:
# CatBoost recibe los datos SIN encoding, solo indicar qué columnas son categóricas
cat_col_names = cat_features  # ['channel', 'device', 'plan', 'country']

X_train_cb = X_train.copy()
X_test_cb  = X_test.copy()
# CatBoost requiere que las categóricas sean string
for c in cat_col_names:
    X_train_cb[c] = X_train_cb[c].astype(str)
    X_test_cb[c]  = X_test_cb[c].astype(str)

cat_model = CatBoostClassifier(
    iterations=1000, learning_rate=0.05,
    depth=6, l2_leaf_reg=3,
    cat_features=cat_col_names,
    eval_metric='AUC',
    early_stopping_rounds=30,
    random_seed=42, verbose=0
)

X_tr_cb, X_val_cb, y_tr_cb, y_val_cb = train_test_split(
    X_train_cb, yc_train, test_size=0.15, random_state=0
)
cat_model.fit(X_tr_cb, y_tr_cb, eval_set=(X_val_cb, y_val_cb))

print(f'Early stopping en ronda: {cat_model.best_iteration_}')
print(f'Test AUC: {roc_auc_score(yc_test, cat_model.predict_proba(X_test_cb)[:,1]):.4f}')
print(f'Test F1:  {f1_score(yc_test, cat_model.predict(X_test_cb)):.4f}')

## 5 — Feature Importance: los tres frameworks

In [ ]:
# XGBoost importance
xgb_final = xgb.XGBClassifier(
    n_estimators=xgb_es.best_iteration, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='logloss', verbosity=0
)
xgb_final.fit(X_train_enc, yc_train)
xgb_imp = pd.Series(xgb_final.feature_importances_, index=feat_names)

# LightGBM importance (gain)
lgb_gain = pd.Series(
    lgb_model.booster_.feature_importance(importance_type='gain'),
    index=feat_names
)

# CatBoost importance
cat_imp = pd.Series(cat_model.feature_importances_, index=feat_names)

# Normalizar para comparar
def normalize(s): return s / s.sum()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#4361ee', '#06d6a0', '#f72585']
for ax, (imp, title), color in zip(axes, [
    (normalize(xgb_imp).sort_values(ascending=False).head(8), 'XGBoost (weight)'),
    (normalize(lgb_gain).sort_values(ascending=False).head(8), 'LightGBM (gain)'),
    (normalize(cat_imp).sort_values(ascending=False).head(8),  'CatBoost'),
], colors):
    ax.barh(imp.index[::-1], imp.values[::-1], color=color, alpha=0.85)
    ax.set_xlabel('Importancia (normalizada)')
    ax.set_title(title)

plt.suptitle('Feature Importances — XGBoost vs LightGBM vs CatBoost', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 — Comparación directa: clasificación y regresión

In [ ]:
# Clasificación: tabla comparativa
results = []
models_clf = [
    ('XGBoost',   xgb_final,  X_test_enc),
    ('LightGBM',  lgb_model,  X_test_enc),
    ('CatBoost',  cat_model,  X_test_cb),
]
print(f'{"Framework":<12} {"TestAUC":>9} {"Acc":>7} {"F1":>7}')
print('-' * 40)
for name, model, X_te in models_clf:
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = model.predict(X_te)
    auc = roc_auc_score(yc_test, y_prob)
    acc = accuracy_score(yc_test, y_pred)
    f1  = f1_score(yc_test, y_pred)
    results.append({'name': name, 'auc': auc})
    print(f'{name:<12} {auc:>9.4f} {acc:>7.4f} {f1:>7.4f}')

print('\n--- Regresión ---')
reg_configs = [
    ('XGBoost', xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5,
                                   subsample=0.8, colsample_bytree=0.8,
                                   random_state=42, verbosity=0), X_train_enc, X_test_enc),
    ('LightGBM', lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=63,
                                    min_child_samples=20, random_state=42, verbose=-1),
     X_train_enc, X_test_enc),
    ('CatBoost', CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6,
                                    cat_features=cat_col_names, random_seed=42, verbose=0),
     X_train_cb, X_test_cb),
]
print(f'{"Framework":<12} {"RMSE":>9} {"R²":>8}')
print('-' * 32)
for name, model, X_tr, X_te in reg_configs:
    y_tr_in = yr_train if name != 'CatBoost' else yr_train
    model.fit(X_tr, yr_train)
    y_pred = model.predict(X_te)
    rmse = mean_squared_error(yr_test, y_pred) ** 0.5
    r2   = r2_score(yr_test, y_pred)
    print(f'{name:<12} ${rmse:>8.1f} {r2:>8.4f}')

## 7 — Integración con sklearn Pipeline (XGBoost / LightGBM)

In [ ]:
from sklearn.model_selection import cross_val_score

# XGBoost y LightGBM son sklearn-compatibles → funcionan directamente en Pipeline
pipe_xgb = Pipeline([
    ('prep', preprocessor_ord),
    ('model', xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.08, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='logloss', verbosity=0
    ))
])

pipe_lgb = Pipeline([
    ('prep', preprocessor_ord),
    ('model', lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.08, num_leaves=63,
        min_child_samples=20, random_state=42, verbose=-1
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipe in [('XGBoost Pipeline', pipe_xgb), ('LightGBM Pipeline', pipe_lgb)]:
    scores = cross_val_score(pipe, X_train, yc_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'{name:<22} CV AUC: {scores.mean():.4f} ± {scores.std():.4f}')

print('\n→ Se puede usar GridSearchCV / Optuna sobre estos pipelines directamente.')
print('  Prefijo de hiperparámetros en GridSearch: model__n_estimators, model__learning_rate, ...')

## Resumen — ¿Cuándo usar cada framework?

| Framework | Fortaleza | Cuándo elegirlo |
|---|---|---|
| **XGBoost** | Robusto, muy documentado, GPU nativo | Default si no hay restricciones |
| **LightGBM** | Muy rápido en datasets grandes (>100k filas) | Cuando el tiempo de entrenamiento importa |
| **CatBoost** | Categóricas nativas, poco tuning necesario | Muchas categóricas de alta cardinalidad, prototipado rápido |

**Hiperparámetros críticos (los tres):**
1. `learning_rate` + `n_estimators` con **early stopping** — siempre
2. Profundidad: `max_depth` (XGB/CB) o `num_leaves` (LGB)
3. Submuestreo: `subsample` + `colsample_bytree` (XGB/LGB) — regularización implícita
4. `min_child_weight` / `min_data_in_leaf` / `min_child_samples` — controla tamaño de hojas
5. Regularización L1/L2: `reg_alpha`, `reg_lambda`

> Para búsqueda eficiente de hiperparámetros ver **`09c_optuna_hyperparameter_tuning.ipynb`**